# 04 · Ventana coincidente: ocupación vs. afluencia (PLAN §3.5 / Módulo 3)

La guía pide, para el Módulo 3, cruzar la afluencia de visitantes con la ocupación
hotelera, limitado estrictamente a los meses en que **ambas series existen** (ventana
coincidente). Para Izabal esa ventana es **septiembre–diciembre 2025**
(`VENTANA_COINCIDENTE` en `src/config.py`): la ocupación termina en 2025-12 y la
encuesta cubre todo ese tramo sin huecos.

Este notebook reproduce la tabla `occupancy_vs_affluence` y responde la pregunta de
fondo: **¿la afluencia medida explica la ocupación hotelera?** La respuesta corta,
medida en `docs/PLAN.md` §3.1, es que la relación es débil (R² ≈ 0.069), y por eso el
pronóstico del Módulo 2 se construye sobre la propia serie de ocupación, no sobre la
afluencia.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from src.etl.clean import clean_migration, clean_occupancy, clean_survey
from src.etl.load import load_migration, load_occupancy, load_survey
from src.etl import transform as t
from src.config import DESTINATION_DEPT, VENTANA_COINCIDENTE

pd.set_option("display.width", 160)

mig = clean_migration(load_migration())
occ = clean_occupancy(load_occupancy())
sur = clean_survey(load_survey())
print("migración:", mig.shape, "| ocupación:", occ.shape, "| encuesta:", sur.shape)
print("Ventana coincidente declarada:", VENTANA_COINCIDENTE)

## A. La ventana coincidente: qué meses tienen ambas series

`occupancy_vs_affluence` cruza, mes a mes, la ocupación de Izabal con varias medidas de
afluencia (muestral y absoluta) y marca con `en_ventana` los meses dentro de
`VENTANA_COINCIDENTE`. Fuera de esa ventana puede haber ocupación sin encuesta, o
encuesta sin ocupación: eso también se conserva, no se recorta en silencio.

In [ ]:
ova = t.occupancy_vs_affluence(occ, sur, mig)
display(ova.round(3))

cobertura = ova.assign(
    tiene_ocupacion=ova["ocupacion"].notna(),
    tiene_encuesta=ova["registros_encuesta"].notna(),
)[["fecha", "tiene_ocupacion", "tiene_encuesta", "en_ventana"]]
print("\nMeses con AMBAS series (no solo la ventana declarada):")
print(cobertura[cobertura["tiene_ocupacion"] & cobertura["tiene_encuesta"]]["fecha"].dt.strftime("%Y-%m").tolist())
print("\nMeses de la ventana declarada:")
print(ova.loc[ova["en_ventana"], "fecha"].dt.strftime("%Y-%m").tolist())

## B. Correlación: la afluencia no explica bien la ocupación

Se prueban varias medidas de afluencia contra la ocupación, con Pearson (relación
lineal) y Spearman (relación monótona, más robusta a outliers). `participacion_izabal_encuesta`
corrige el tamaño variable de la muestra mensual de la encuesta; las demás son conteos
o proporciones sin esa corrección.

**Regla aplicada:** nunca se grafica la migración (personas, absoluto) en el mismo eje
numérico que un índice [0,1] sin normalizar — es la misma lógica de la regla 5 (no
mezclar aeronaves con personas): magnitudes distintas, ejes distintos.

In [ ]:
MEDIDAS = {
    "participacion_izabal_encuesta": "Participación de Izabal en la encuesta (corrige tamaño de muestra)",
    "registros_encuesta": "Encuestas con destino Izabal (n, muestral)",
    "visitantes_encuesta": "Visitantes en la muestra",
    "proporcion_pernocta": "% de visitantes que pernocta",
    "estadia_ponderada": "Estadía media ponderada (noches)",
    "migracion_alimentadoras": "Visitantes por fronteras alimentadoras (migración, absoluto)",
    "transito_izabal": "Tránsito marítimo en Izabal ('Viajero', migración)",
}

filas = []
for col, etiqueta in MEDIDAS.items():
    pares = ova.dropna(subset=["ocupacion", col])
    if len(pares) < 3 or pares[col].nunique() < 2:
        filas.append({"medida": etiqueta, "n": len(pares), "pearson_r": np.nan,
                      "pearson_p": np.nan, "spearman_rho": np.nan, "spearman_p": np.nan})
        continue
    r, p = stats.pearsonr(pares[col], pares["ocupacion"])
    rho, p_s = stats.spearmanr(pares[col], pares["ocupacion"])
    filas.append({"medida": etiqueta, "n": len(pares), "pearson_r": r, "pearson_p": p,
                  "r2": r ** 2, "spearman_rho": rho, "spearman_p": p_s})

tabla_corr = pd.DataFrame(filas)
display(tabla_corr.round(3))
print("\nTodas las relaciones son débiles (|r| por debajo de ~0.3) salvo que el dato")
print("lo contradiga en esta corrida: eso es lo que hay que verificar aquí, no asumir.")

## C. Visualización: dos paneles con su propia escala

Se grafica la ocupación y la medida de afluencia elegida en **paneles separados**
(nunca con doble eje), resaltando la ventana coincidente. Los huecos de la encuesta
(p. ej. 2025-01) se marcan como ausencia, no como cero.

In [ ]:
MEDIDA_GRAFICO = "participacion_izabal_encuesta"
V_INI, V_FIN = (pd.Timestamp(x) for x in VENTANA_COINCIDENTE)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Ocupación hotelera de Izabal", MEDIDAS[MEDIDA_GRAFICO]))
fig.add_scatter(x=ova["fecha"], y=ova["ocupacion"], mode="lines+markers", name="Ocupación", row=1, col=1)
fig.add_bar(x=ova["fecha"], y=ova[MEDIDA_GRAFICO], name=MEDIDAS[MEDIDA_GRAFICO], row=2, col=1)
for r in (1, 2):
    fig.add_vrect(x0=V_INI, x1=V_FIN, fillcolor="teal", opacity=0.08, line_width=0, row=r, col=1)
fig.update_yaxes(tickformat=".0%", range=[0, 1], row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(height=520, showlegend=False,
                  title="Ocupación y afluencia avanzan por separado: no comparten pico")
fig.show()

## D. Por qué el pronóstico del Módulo 2 no usa la afluencia como driver

Con una relación tan débil entre afluencia medida y ocupación (R² ≈ 0.069 sobre el
cruce completo, por debajo del umbral que justificaría usarla como variable
explicativa principal), el pronóstico de ocupación de Izabal se construye sobre **la
propia serie de ocupación** (Logit-ETS, `src/models/forecast.py`), no sobre un modelo
de regresión con la afluencia como insumo.

La migración **sí** se usa, pero como insumo del método de la guía (`factor_reparto`):
un perfil estacional de referencia, no como predictor lineal de la ocupación.

Esto no es una limitación oculta: se reporta en el dashboard (Módulo 3, sección de
ventana coincidente) y en `docs/memoria_tecnica.md` §5 como un hallazgo medido, no como
un supuesto.

## E. Qué significa para INGUAT

1. **No basta con medir afluencia para explicar ocupación.** Factores no observados en
   estas tres fuentes (precio, capacidad instalada, estacionalidad de la oferta) pesan
   más que el volumen de visitantes medido por la encuesta o la migración.
2. **La recomendación de incorporar el inventario hotelero** (establecimientos,
   habitaciones, plazas cama — ver `docs/catalogo_indicadores.md`, nota del Módulo 3) no
   es solo para completar un campo de la guía: sin capacidad instalada no se puede saber
   si la ocupación sube porque hay más demanda o porque hay menos oferta disponible.